<a href="https://colab.research.google.com/github/CMILINAZZO/medical-llm-self-bias-audit/blob/main/notebooks/02b_generation_local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment Setup & Installations
# We need to install specific libraries for local GPU inference, including accelerate and bitsandbytes for 4-bit quantization to fit these models into Colab's limited VRAM.
!pip install -q transformers accelerate bitsandbytes datasets pandas

In [ ]:
# Cell 2: Secure Credentials & Hugging Face Login
import os
from google.colab import userdata
from huggingface_hub import login

# We require a Hugging Face token to download gated models like Llama 3
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

print("✓ Successfully authenticated with Hugging Face Hub.")

In [ ]:
# Cell 3: GPU Verification & Path Setup
import torch
import pandas as pd

# Verify GPU is active
if not torch.cuda.is_available():
    raise SystemError("❌ CRITICAL: GPU not detected. Please switch Colab runtime to T4 GPU.")
else:
    print(f"✓ GPU Detected: {torch.cuda.get_device_name(0)}")

# Load the baseline data
DATA_PATH = "../data/raw_baseline.csv"
OUTPUT_PATH = "../outputs/generation_results_local.csv"

df = pd.read_csv(DATA_PATH)
print(f"✓ Core generation DataFrame initialized with {df.shape[0]} rows.")

# System Prompt
SYSTEM_PROMPT = """You are an advanced medical student analyzing clinical literature.
Your task is to answer the provided clinical question based strictly on the provided context.
Provide a clear, cohesive, and professional medical rationale followed by your definitive conclusion."""

In [ ]:
# Cell 4: Initialize Llama 3 with Quantization
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

llama_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Configure 4-bit quantization to prevent memory crashes
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"🚀 Loading {llama_model_id} into VRAM...")
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_id)
llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
print("✓ Llama 3 successfully loaded.")

In [ ]:
# Cell 5: Initialize Gemma 3 (or current Gemma instruction model)
gemma_model_id = "google/gemma-7b-it"

print(f"🚀 Loading {gemma_model_id} into VRAM...")
gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_model_id)
gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
print("✓ Gemma successfully loaded.")

In [ ]:
# Cell 6: Local Generation Loop
from tqdm import tqdm

llama_responses = []
gemma_responses = []

print("🚀 Starting local generation loop for open-weights models...")

for idx, row in tqdm(df.iterrows(), total=df.shape[0], desc="Generating Local Responses"):
    context = row['context']
    question = row['question']

    prompt = f"Context:\n{context}\n\nQuestion:\n{question}"

    # --- Llama 3 Generation ---
    llama_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    llama_inputs = llama_tokenizer.apply_chat_template(llama_messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    llama_outputs = llama_model.generate(llama_inputs, max_new_tokens=512, temperature=0.2, do_sample=True)
    llama_responses.append(llama_tokenizer.decode(llama_outputs[0][llama_inputs.shape[-1]:], skip_special_tokens=True))

    # --- Gemma Generation ---
    gemma_messages = [
        {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{prompt}"} # Gemma prefers user-prompt heavy instructions
    ]
    gemma_inputs = gemma_tokenizer.apply_chat_template(gemma_messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    gemma_outputs = gemma_model.generate(gemma_inputs, max_new_tokens=512, temperature=0.2, do_sample=True)
    gemma_responses.append(gemma_tokenizer.decode(gemma_outputs[0][gemma_inputs.shape[-1]:], skip_special_tokens=True))

# Attach to local dataframe
df_local = df.copy()
df_local['response_llama'] = llama_responses
df_local['response_gemma'] = gemma_responses

import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_local.to_csv(OUTPUT_PATH, index=False)
print(f"\n✓ Local generation complete. Saved to: {OUTPUT_PATH}")